# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is defined via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# `dataset.metadata` is a metadata object, so get its JSON representation
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}\n{metadata_json['description']}")


## 2. Data Overview
Review available record sets, fields, and their IDs.
This step allows us to inspect the main structural entities in the dataset, referencing them by their unique `@id` identifiers.

In [ ]:
# Enumerate record sets, fields, columns, referenced by @id
record_sets = dataset.metadata.record_sets

print('Record Sets in dataset:')
for rs in record_sets:
    print(f"- (@id): {rs['@id']}, name: {rs.get('name', '[no name]')}")
    
    fields = rs.get('field', [])
    if fields:
        print("  Fields:")
        for f in fields:
            print(f"    - (@id): {f['@id']}, name: {f.get('name', '[no name]')}")

    columns = rs.get('column', [])
    if columns:
        print("  Columns:")
        for c in columns:
            print(f"    - (@id): {c['@id']}, name: {c.get('name', '[no name]')}")
    print()

# For demonstration, select the first record set
first_record_set_id = record_sets[0]['@id'] if record_sets else None
if first_record_set_id:
    print(f"First record set @id: {first_record_set_id}")

## 3. Data Extraction
Load data from the primary record set into a DataFrame for analysis.
All references use the unique `@id` fields for record sets and columns.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"No records found for record set {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(3))

# Select the first record set loaded for further analysis
main_df_id = list(dataframes.keys())[0] if dataframes else None
if main_df_id:
    print(f"Example DataFrame columns for {main_df_id}: {dataframes[main_df_id].columns.tolist()}")
    display(dataframes[main_df_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, grouping, and outlier removal.
- All field references use their `@id`s.
- We'll select a numeric column, filter rows, normalize it, and group by another categorical field.

In [ ]:
# EDA: Select numeric and categorical fields by @id
if main_df_id:
    df = dataframes[main_df_id]

    # Find a numeric column: Assume one is named 'gad7_score' or similar
    numeric_col_candidates = [c for c in df.columns if 'score' in c or 'phq9' in c or 'psq' in c or 'age' in c]
    numeric_field_id = numeric_col_candidates[0] if numeric_col_candidates else df.columns[0]

    print(f"Using numeric field (@id): {numeric_field_id}")

    # Set a threshold for demonstration
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping - try grouping by 'gender' or 'level_of_education' if present
    group_field_candidates = [c for c in df.columns if 'gender' in c or 'education' in c or 'marital' in c]
    group_field_id = group_field_candidates[0] if group_field_candidates else df.columns[0]
    print(f"Grouping by field (@id): {group_field_id}")
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
- We'll plot the distribution of the selected numeric field and a grouped bar chart for the grouping field.
- All field references use their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df_id:
    df = dataframes[main_df_id]

    # Plot distribution of numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Plot group-wise mean scores
    if 'grouped_df' in locals():
        plt.figure(figsize=(8, 5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, palette='viridis')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated:
- How to load and access FAIR² survey data using the `mlcroissant` library
- Exploration and referencing of record sets and fields by their `@id`
- Data extraction, filtering, normalization, grouping, and visualization

Key findings:
- The dataset contains nuanced survey responses including scores for mental health indicators
- Group-wise averages can highlight differences across demographic groups
- Data is accessible, well-structured, and ready for further AI/ML analyses

Please cite according to: Mugotitsa et al. 2026, Frontiers ([full citation in dataset metadata]).
